In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torchvision.models as models

In [ ]:
# 0. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

########################################################
# 1. 학습/검증 데이터 전처리 파이프라인 분리 (v2 적용)
########################################################

# 1-1. 학습 데이터용 파이프라인 (ToImage 추가)
train_transform = transforms.Compose([
    transforms.ToImage(),                                # PIL 이미지를 Image Tensor로 변환
    transforms.Resize((224, 224)),                       # 224x224 리사이징
    transforms.Grayscale(num_output_channels=3),        # 1채널 -> 3채널(RGB) 변환
    transforms.RandomHorizontalFlip(p=0.5),             # 50% 확률로 좌우 반전
    transforms.RandomRotation(degrees=15),               # ±15도 범위 내 랜덤 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2),# Brightness/Contrast 변형
    transforms.ToDtype(torch.float32, scale=True),      # float32 변환 및 0~1 스케일링
    transforms.Normalize(                               # ImageNet 사전 학습 모델 표준 정규화
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# 1-2. 검증 및 테스트 데이터용 파이프라인 (ToImage 추가)
val_test_transform = transforms.Compose([
    transforms.ToImage(),                                # PIL 이미지를 Image Tensor로 변환
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
########################################################
# 2. 데이터셋 분할 및 Transform 각각 적용
########################################################

# Subset 개별 transform 적용을 위한 커스텀 Dataset 클래스
class TransformedDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# 전체 데이터 로드 (원본 상태)
full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=val_test_transform)

# Train / Validation 분할 (55,000 / 5,000)
train_size = 55000
val_size = 5000
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# 각 Split에 맞는 Transform 지정
train_dataset = TransformedDataset(train_subset, transform=train_transform)
val_dataset = TransformedDataset(val_subset, transform=val_test_transform)

# 데이터로더 구축
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets
import torchvision.transforms.v2 as transforms
import torchvision.models as models
import copy

# 0. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

########################################################
# 1. 전처리 파이프라인 정의 (전 / 후)
########################################################

# [전처리 전] - 기본 전처리만 적용 (ToImage + Resize + Grayscale + ToDtype)
transform_before = transforms.Compose([
    transforms.ToImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToDtype(torch.float32, scale=True),
])

# [전처리 후] - 데이터 증강(Augmentation) + 정규화(Normalize) 추가
transform_after = transforms.Compose([
    transforms.ToImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),             # 좌우 반전
    transforms.RandomRotation(degrees=15),               # 랜덤 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2),# 색상 변형
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(                               # ImageNet 표준 정규화
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# [검증 및 테스트용] - 평가 시에는 증강을 제외하고 동일한 스케일 적용
val_test_transform_before = transform_before

val_test_transform_after = transforms.Compose([
    transforms.ToImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

########################################################
# 2. 데이터셋 및 Subset 래퍼 설정
########################################################

class TransformedDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# 원본 데이터 로드
full_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True)
test_dataset_raw = datasets.FashionMNIST(root='./data', train=False, download=True)

# 학습/검증 데이터 고정 분할
train_size, val_size = 55000, 5000
generator = torch.Generator().manual_seed(42) # 동일한 분할을 위한 시드 고정
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

########################################################
# 3. 모델 생성 함수 및 공통 시드 설정
########################################################

def get_resnet_model():
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 10)
    return model.to(device)

# 초기 모델 가중치 동일하게 보장
torch.manual_seed(42)
initial_model = get_resnet_model()

########################################################
# 4. 학습 & 평가 전용 함수
########################################################

def train_and_eval(model, train_loader, val_loader, test_loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Acc: {(train_correct/train_total)*100:.2f}% | "
              f"Val Acc: {(val_correct/val_total)*100:.2f}%")

    # --- Test ---
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()
            
    test_acc = (test_correct / test_total) * 100
    return test_acc

########################################################
# 5. [실험 1] 전처리 적용 전 (Basic Pipeline)
########################################################
print("\n==========================================")
print(" 1. 전처리(데이터 증강/정규화) [적용 전] 학습 시작")
print("==========================================")

train_loader_before = DataLoader(TransformedDataset(train_subset, transform_before), batch_size=64, shuffle=True)
val_loader_before = DataLoader(TransformedDataset(val_subset, val_test_transform_before), batch_size=64, shuffle=False)
test_loader_before = DataLoader(TransformedDataset(test_dataset_raw, val_test_transform_before), batch_size=64, shuffle=False)

model_before = copy.deepcopy(initial_model) # 동일한 가중치 복사
acc_before = train_and_eval(model_before, train_loader_before, val_loader_before, test_loader_before, epochs=5)

########################################################
# 6. [실험 2] 전처리 적용 후 (Augmentation + Normalization)
########################################################
print("\n==========================================")
print(" 2. 전처리(데이터 증강/정규화) [적용 후] 학습 시작")
print("==========================================")

train_loader_after = DataLoader(TransformedDataset(train_subset, transform_after), batch_size=64, shuffle=True)
val_loader_after = DataLoader(TransformedDataset(val_subset, val_test_transform_after), batch_size=64, shuffle=False)
test_loader_after = DataLoader(TransformedDataset(test_dataset_raw, val_test_transform_after), batch_size=64, shuffle=False)

model_after = copy.deepcopy(initial_model) # 동일한 가중치 복사
acc_after = train_and_eval(model_after, train_loader_after, val_loader_after, test_loader_after, epochs=5)

########################################################
# 7. 최종 성능 비교 출력
########################################################
print("\n==========================================")
print(" 최종 성능 비교 결과")
print("==========================================")
print(f"• 전처리 적용 전 Test Accuracy : {acc_before:.2f}%")
print(f"• 전처리 적용 후 Test Accuracy : {acc_after:.2f}%")
print(f"• 성능 변화 (증감폭)           : {acc_after - acc_before:+.2f}%p")
print("==========================================")